# TAREA 3: PRIMERA PARTE 
## Sandro Sotacuro Escobar
# Brecha Digital Territorial — Cusco, Perú
## Análisis Ráster Geoespacial
**Datos:** NASA VNL 2025 (EPSG:4326) · OSIPTEL Kernel Móvil 2019 50m (EPSG:32719)

---
## Paso 0 — Configuración del entorno

In [1]:
# Instalación de dependencias
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "rasterio", "numpy", "matplotlib", "scipy", "seaborn", "pandas",
                "--break-system-packages", "-q"], capture_output=True)

import rasterio
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy
import seaborn as sns
import pandas as pd
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import from_bounds
from scipy.ndimage import gaussian_filter
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print(f"rasterio  {rasterio.__version__}")
print(f"numpy     {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"scipy     {scipy.__version__}")
print(f"seaborn   {sns.__version__}")
print(f"pandas    {pd.__version__}")

rasterio  1.5.0
numpy     2.4.4
matplotlib 3.10.9
scipy     1.17.1
seaborn   0.13.2
pandas    3.0.2


In [2]:
import os
# Rutas de datos — el notebook vive en notebooks/, los datos en data/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('digital_divide_cusco.ipynb'))
BASE_DIR     = os.path.dirname(NOTEBOOK_DIR)   # sube un nivel: raster-digital-divide/
DATA_DIR     = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR   = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

VNL_PATH  = os.path.join(DATA_DIR, 'VNL_cusco_2025.tif')
CONN_PATH = os.path.join(DATA_DIR, 'kernel_cobmovil2019_50m.tif')

# Verificar existencia
for p in [VNL_PATH, CONN_PATH]:
    status = '✓' if os.path.exists(p) else '✗ FALTA'
    print(f"{status}  {p}")

✓  d:\Data_science_Python_2026\raster-digital-divide\data\VNL_cusco_2025.tif
✓  d:\Data_science_Python_2026\raster-digital-divide\data\kernel_cobmovil2019_50m.tif


---
## Paso 1 — Carga e inspección de rásteres

In [3]:
def inspect_raster(path, label):
    """Imprime metadatos completos de un ráster."""
    with rasterio.open(path) as src:
        data = src.read(1)
        nodata = src.nodata
        res_x, res_y = src.res
        # Resolución aproximada en km (1° ≈ 111 km en ecuador)
        res_km_x = abs(res_x) * 111
        res_km_y = abs(res_y) * 111

        # Máscara de píxeles válidos
        if nodata is not None:
            valid_mask = ~np.isclose(data, nodata, rtol=1e-5, atol=1e30)
        else:
            valid_mask = np.isfinite(data)
        valid = data[valid_mask]

        print(f"{'='*55}")
        print(f" {label}")
        print(f"{'='*55}")
        print(f"  CRS              : {src.crs}")
        print(f"  Forma (alto×ancho): {src.height} × {src.width}")
        print(f"  Bandas            : {src.count}")
        print(f"  NoData            : {nodata}")
        print(f"  Tipo de dato      : {src.dtypes[0]}")
        print(f"  Extensión (bounds): {src.bounds}")
        print(f"  Resolución px     : {res_x:.6f}° × {res_y:.6f}°")
        print(f"  Resolución aprox  : {res_km_x:.3f} km × {res_km_y:.3f} km")
        print(f"  Píxeles válidos   : {valid_mask.sum():,}")
        print(f"  Rango de valores  : [{valid.min():.6f}, {valid.max():.6f}]")
        print()
    return

inspect_raster(VNL_PATH,  'VNL_cusco_2025.tif  — Luces nocturnas NASA (nW·cm⁻²·sr⁻¹)')
inspect_raster(CONN_PATH, 'kernel_cobmovil2019_50m.tif — Densidad kernel cobertura móvil')

 VNL_cusco_2025.tif  — Luces nocturnas NASA (nW·cm⁻²·sr⁻¹)
  CRS              : EPSG:4326
  Forma (alto×ancho): 1081 × 961
  Bandas            : 1
  NoData            : None
  Tipo de dato      : float32
  Extensión (bounds): BoundingBox(left=-74.00208248534999, bottom=-15.50208405735, right=-69.99791578664998, top=-10.99791735465)
  Resolución px     : 0.004167° × 0.004167°
  Resolución aprox  : 0.463 km × 0.463 km
  Píxeles válidos   : 1,038,841
  Rango de valores  : [-1.500000, 1254.614502]

 kernel_cobmovil2019_50m.tif — Densidad kernel cobertura móvil
  CRS              : EPSG:32719
  Forma (alto×ancho): 6116 × 7754
  Bandas            : 1
  NoData            : -3.4028234663852886e+38
  Tipo de dato      : float32
  Extensión (bounds): BoundingBox(left=-43080.11101302641, bottom=8337100.058809407, right=344619.88898697356, top=8642900.058809407)
  Resolución px     : 50.000000° × 50.000000°
  Resolución aprox  : 5550.000 km × 5550.000 km
  Píxeles válidos   : 47,423,464
  Rango de

---
## Paso 2: Reproyección y alineación de la cuadrícula

In [4]:
# --- Leer metadatos de VNL (referencia) ---
with rasterio.open(VNL_PATH) as src_vnl:
    vnl_crs       = src_vnl.crs
    vnl_transform = src_vnl.transform
    vnl_height    = src_vnl.height
    vnl_width     = src_vnl.width
    vnl_raw       = src_vnl.read(1).astype(np.float32)

# --- Reproyectar + remuestrear conectividad a cuadrícula VNL ---
conn_aligned = np.zeros((vnl_height, vnl_width), dtype=np.float32)

with rasterio.open(CONN_PATH) as src_conn:
    reproject(
        source      = rasterio.band(src_conn, 1),
        destination = conn_aligned,
        src_transform = src_conn.transform,
        src_crs       = src_conn.crs,
        src_nodata    = src_conn.nodata,
        dst_transform = vnl_transform,
        dst_crs       = vnl_crs,
        dst_nodata    = 0.0,
        resampling    = Resampling.bilinear
    )

print("✓ Reproyección y alineación completadas")
print(f"  VNL shape       : {vnl_raw.shape}")
print(f"  Conectividad shape: {conn_aligned.shape}")
assert vnl_raw.shape == conn_aligned.shape, "¡Shapes no coinciden!"
print("  Verificación de dimensiones: OK ✓")

✓ Reproyección y alineación completadas
  VNL shape       : (1081, 961)
  Conectividad shape: (1081, 961)
  Verificación de dimensiones: OK ✓


---
## Paso 3: Normalización robusta

In [5]:
def robust_normalize(arr, nodata_val=None, p_low=2, p_high=98):
    """
    Normalización percentil [p_low, p_high] → [0, 1].
    Reemplaza negativos y NoData con 0 antes de recortar.
    """
    out = arr.copy().astype(np.float32)
    # Reemplazar NoData
    if nodata_val is not None:
        out[np.isclose(out, nodata_val, rtol=1e-5, atol=1e28)] = 0.0
    # Reemplazar negativos
    out[out < 0] = 0.0
    # Reemplazar NaN/Inf
    out = np.where(np.isfinite(out), out, 0.0)

    plo = np.percentile(out[out > 0], p_low)  if np.any(out > 0) else 0.0
    phi = np.percentile(out[out > 0], p_high) if np.any(out > 0) else 1.0

    if phi == plo:
        return np.zeros_like(out)

    norm = (out - plo) / (phi - plo)
    norm = np.clip(norm, 0, 1)
    return norm


# Obtener nodata del ráster de conectividad
with rasterio.open(CONN_PATH) as src:
    conn_nodata = src.nodata

vnl_norm  = robust_normalize(vnl_raw,       nodata_val=None)
conn_norm = robust_normalize(conn_aligned,   nodata_val=conn_nodata)

for name, arr in [('VNL normalizado', vnl_norm), ('Conectividad normalizada', conn_norm)]:
    print(f"{name}:")
    print(f"  min={arr.min():.4f}  max={arr.max():.4f}  "
          f"media={arr.mean():.4f}  std={arr.std():.4f}")

VNL normalizado:
  min=0.0000  max=1.0000  media=0.0035  std=0.0414
Conectividad normalizada:
  min=0.0000  max=1.0000  media=0.0142  std=0.0807
